In [39]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added to PYTHONPATH:", PROJECT_ROOT)


Project root added to PYTHONPATH: c:\Users\acer\OneDrive\Desktop\iitbbsr


In [ ]:
import json
from tasks.task1_explain import task1_run, load_conversations
from context.context_manager import ContextManager
from tasks.task2_multiturn import answer_followup
from evaluation.id_recall import id_recall_from_explanation
from evaluation.faithfulness import faithfulness

In [40]:
from tasks.task1_explain import task1_run, load_conversations

DATA_PATH = "../data/processed/normalized_conversations.json"

conversations = load_conversations(DATA_PATH)

print(f"Loaded {len(conversations)} normalized conversations")

Loaded 5037 normalized conversations


In [41]:
import json
query = "Why did escalation occur?"

outcome, explanation = task1_run(
    query=query,
    data_path=DATA_PATH,
    use_embeddings=False  # deterministic mode
)

print("=== TASK 1 OUTPUT ===")
print("Outcome:", outcome)
print(json.dumps(explanation.model_dump(), indent=2))


=== TASK 1 OUTPUT ===
Outcome: escalation - repeated service failures
{
  "outcome": "escalation - repeated service failures",
  "call_ids": [
    "7034-5430-2980-5483",
    "2801-6816-9124-1355",
    "3883-8555-9747-5746",
    "4967-8141-1813-8658",
    "3631-7868-1079-2880",
    "5405-1959-8150-2620",
    "6442-9317-2833-5263",
    "4471-5440-7427-7835",
    "6338-1986-7723-6070",
    "2717-1088-7257-7370",
    "9319-3940-9382-2387",
    "4999-7266-1895-5583",
    "3925-2095-7371-7901",
    "8584-4627-1759-3846",
    "8625-9811-3321-2163",
    "3925-3887-6255-8171",
    "3263-1585-8501-2701",
    "7111-8235-8458-4031",
    "9361-3226-5280-7468",
    "3578-4645-2490-5752",
    "9036-1648-4300-2522",
    "5186-2494-1700-6696",
    "7888-5665-2083-7600",
    "7552-6299-5982-6654",
    "1388-4256-8564-5555",
    "7946-2197-7898-4045",
    "9295-9855-5231-5046",
    "3528-5080-2778-7765",
    "6464-2429-2502-2616",
    "6625-3264-7615-2421",
    "2990-4204-4258-9488",
    "3853-4509-5928-

In [42]:
from context.context_manager import ContextManager
context_manager = ContextManager()
state = context_manager.initialize(explanation)

print("=== CONTEXT INITIALIZED ===")
print("Active Outcome:", state.active_outcome)
print("Active Call IDs:", state.active_call_ids)
print("Pinned Evidence Count:", len(state.pinned_evidence))


=== CONTEXT INITIALIZED ===
Active Outcome: escalation - repeated service failures
Active Call IDs: ['7034-5430-2980-5483', '2801-6816-9124-1355', '3883-8555-9747-5746', '4967-8141-1813-8658', '3631-7868-1079-2880', '5405-1959-8150-2620', '6442-9317-2833-5263', '4471-5440-7427-7835', '6338-1986-7723-6070', '2717-1088-7257-7370', '9319-3940-9382-2387', '4999-7266-1895-5583', '3925-2095-7371-7901', '8584-4627-1759-3846', '8625-9811-3321-2163', '3925-3887-6255-8171', '3263-1585-8501-2701', '7111-8235-8458-4031', '9361-3226-5280-7468', '3578-4645-2490-5752', '9036-1648-4300-2522', '5186-2494-1700-6696', '7888-5665-2083-7600', '7552-6299-5982-6654', '1388-4256-8564-5555', '7946-2197-7898-4045', '9295-9855-5231-5046', '3528-5080-2778-7765', '6464-2429-2502-2616', '6625-3264-7615-2421', '2990-4204-4258-9488', '3853-4509-5928-6824', '9065-7143-5542-3940', '3425-7410-3440-7404', '3605-5214-9667-1936', '8144-4002-9143-6860', '7522-7011-4390-3472', '8167-2369-5496-4527', '5769-2119-7686-5558', '3

In [43]:
from tasks.task2_multiturn import answer_followup
followup_queries = [
    "Which factors contributed?",
    "Which turns caused escalation?",
    "Why did this happen?"
]

print("=== TASK 2 FOLLOW-UPS ===")

for q in followup_queries:
    context_manager.log_query(q, "followup")
    response = answer_followup(state, q)
    print(f"\nFOLLOW-UP QUESTION: {q}")
    print(json.dumps(response, indent=2))


=== TASK 2 FOLLOW-UPS ===

FOLLOW-UP QUESTION: Which factors contributed?
{
  "category": "factors",
  "active_outcome": "escalation - repeated service failures",
  "factors": [
    {
      "name": "customer_frustration",
      "description": "Customer expresses frustration or dissatisfaction, often indicating service failure or unresolved issues.",
      "evidence_count": 2688
    },
    {
      "name": "repetition_without_resolution",
      "description": "Customer indicates they repeated the issue multiple times without resolution, increasing escalation likelihood.",
      "evidence_count": 1443
    },
    {
      "name": "lack_of_proactive_communication",
      "description": "Customer indicates they were not notified / informed, which can trigger blame and escalation.",
      "evidence_count": 490
    },
    {
      "name": "high_stakes_urgency",
      "description": "Time-sensitive or high-impact consequences raise emotional intensity and escalation risk.",
      "evidence_count"

In [44]:

from evaluation.id_recall import id_recall_from_explanation
gold_call_ids = [
    c.call_id for c in conversations if c.outcome == outcome
]

idrecall = id_recall_from_explanation(explanation, gold_call_ids)

print("=== EVALUATION ===")
print("IDRecall:", idrecall)


=== EVALUATION ===
IDRecall: 1.0


In [45]:
from evaluation.faithfulness import faithfulness
faith = faithfulness(explanation, conversations)

print("Faithfulness:", faith)


Faithfulness: 1.0


In [46]:
print("Notebook execution completed successfully.")


Notebook execution completed successfully.


In [48]:

from evaluation.relevancy import relevancy


rel = relevancy(state, user_query, system_response)


print("Relevancy:", rel)


Relevancy: 1.0
